In [1]:
#importing the Libraies
import numpy as np
import pandas as pd

In [2]:
# Reading the Dataset
dataset = pd.read_csv('insurance_pre.csv')

In [3]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [5]:
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [6]:
indep=dataset[['age', 'bmi', 'children','sex_male', 'smoker_yes']]
dep=dataset[['charges']]

In [7]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 0.20, random_state = 0)

In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
from sklearn.metrics import r2_score, make_scorer,mean_absolute_error, mean_squared_error

# Step 1: Define Adjusted R² scorer
def make_adjusted_r2_scorer(X):
    n, p = X.shape
    def adjusted_r2(y_true, y_pred):
        r2 = r2_score(y_true, y_pred)
        return 1 - (1 - r2) * ((n - 1) / (n - p - 1))
    return make_scorer(adjusted_r2, greater_is_better=True)

# Step 2: Create custom scorer using your training data
adjusted_r2_scorer = make_adjusted_r2_scorer(X_train)

In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
# ------------------ Random Forest Regressor ------------------
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'criterion': ['squared_error', 'absolute_error']
}

grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
# fitting the model for grid search
grid.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'absolute_error'],
                         'max_depth': [None, 10, 20],
                         'n_estimators': [50, 100, 200]},
             scoring='r2')

In [14]:
# Assuming 'grid' is your GridSearchCV object and has been fitted to the data
print("Best parameters:", grid.best_params_)

Best parameters: {'criterion': 'absolute_error', 'max_depth': 10, 'n_estimators': 100}


In [15]:
# Step 6: Evaluate on test data
y_pred = grid.predict(X_test)

r2 = r2_score(y_test, y_pred)
n, p = X_test.shape
adjusted_r2_test = 1 - (1 - r2) * ((n - 1) / (n - p - 1))

# MAE and MSE
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("Test R²:", r2)
print("Test Adjusted R²:", adjusted_r2_test)
print("Test MAE:", mae)
print("Test MSE:", mse)

Test R²: 0.8843771889552344
Test Adjusted R²: 0.8821706467597237
Test MAE: 2333.371429001492
Test MSE: 18399053.5651318


In [16]:
#In the above the R2 is 88.43% and in the adjusted R2 score it is 88.21% which is quite good...also the MAE is better,MSE wants to be improved by RMSE.